# 03 — Prepare the training data (clean → deduplicate → split → tokenize, once)

Output: one set of token files per source, shared by **all five architecture notebooks**, so every model trains on exactly the same tokens.

| Source                | What we keep                        | Why                                                           |
| --------------------- | ----------------------------------- | ------------------------------------------------------------- |
| `sc` Supreme Court    | everything (≥ 50 words)             | closest to the test style                                     |
| `tribunals`           | everything (≥ 50 words)             | judgment style, similar to SC                                 |
| `acts`                | everything (≥ 50 words)             | legal vocabulary (statutes)                                   |
| `hc_long` High Courts | only documents with **≥ 640 words** | 85% of HC documents are short procedural orders (exploration) |

For each source the notebook:

1. **downloads** the raw file once (fast: ~40 MB/s; streaming it was ~1 MB/s in the exploration),
2. **streams** it record by record from disk with `ijson` (constant RAM, even for the 10.7 GB High Court file),
3. **cleans** each document with `clean_text` (identical to `02_tokenizer`),
4. **removes exact duplicates** across all sources (MD5 fingerprint),
5. **splits** by document: ~1% validation, 99% training,
6. **tokenizes** with `bpe32k`, adds `<|endoftext|>` after each document, and appends to a `uint16` `.bin` file.

Finally it builds a **new local evaluation set** (200 × 512-word prompt + 128-word truth) from cleaned SC validation documents.

**Rules followed:** progress bar for every download and every source; each finished source is **skipped on re-runs** (so if the notebook stops, only the unfinished source is redone); files are written to `.tmp` and renamed only when complete.


## 0. Setup


In [1]:
import os, json, re, unicodedata, random, hashlib, time
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import ijson
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

REPO_ID   = "Exploration-Lab/CS-781-Capstone"
TOK_PATH  = PROJECT_ROOT / "data" / "tokenizer" / "bpe32k.json"
OUT_DIR   = PROJECT_ROOT / "data" / "tokens_bpe32k"          # token files for all architectures
EVAL_DIR  = PROJECT_ROOT / "data" / "local_eval"             # new local evaluation set
OUT_DIR.mkdir(parents=True, exist_ok=True); EVAL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_PER_MILLE = 10            # 10 per 1000 documents = 1% validation
BATCH_DOCS = 256              # documents tokenized together (Rust runs them in parallel)

tokenizer = Tokenizer.from_file(str(TOK_PATH))
EOS_ID = tokenizer.token_to_id("<|endoftext|>")
assert tokenizer.get_vocab_size() <= 65535, "uint16 can only hold ids up to 65535"
print("Project root:", PROJECT_ROOT)
print("Tokenizer   :", TOK_PATH.name, "| vocab", tokenizer.get_vocab_size(), "| EOS id", EOS_ID)
print("Output      :", OUT_DIR)

c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project
Tokenizer   : bpe32k.json | vocab 32000 | EOS id 0
Output      : c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\data\tokens_bpe32k


## 1. The cleaning function — identical copy of `02_tokenizer`

⚠️ If you ever change it, change it in `02_tokenizer`, here, and in every architecture notebook (they clean prompts before generating).


In [2]:
SIGNATURE_RE = re.compile(r"Signature Not Verified.{0,150}?IST(\s*Reason:)?", flags=re.DOTALL)

ASCII_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',
    "\u2013": "-", "\u2014": "-", "\u2015": "-", "\u2212": "-",
    "\u2026": "...",
    "\u00a0": " ", "\u2009": " ", "\u200b": "",
    "\u00ad": "",
    "\u20b9": "Rs.", "\u00a7": "S.", "\u2022": "-", "\u00b7": ".",
})

def clean_text(text):
    text = SIGNATURE_RE.sub(" ", text)
    text = text.translate(ASCII_MAP)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return " ".join(text.split())

print(clean_text("Test\u00ad \u201cquote\u201d\n\n done"))

Test "quote" done


## 2. The processing function (one source → train/val token files)

Key ideas in the code:

- **Early filter:** for `hc_long` we check the record's own `word_count` _before_ cleaning, so the ~85% short orders cost almost nothing.
- **Fingerprint** = the first 8 bytes of the MD5 hash of the cleaned text, stored as one 64-bit integer. Same text → same fingerprint. A `set` of fingerprints finds duplicates in O(1) per document.
- **Hash-based split:** a document goes to validation if `fingerprint % 1000 < 10`. The decision depends only on the text itself, not on file order or on how far we got — so it is **reproducible** and a document can never end up on both sides.
- **Streaming write:** each batch of token ids is appended to the `.bin` file with `arr.tobytes()`, so RAM stays small no matter how big the source is.
- **Progress bar** counts bytes of the raw JSON file read so far (`f.tell()`), so it shows a real ETA.
- **Fingerprints of each finished source are saved** (`*_hashes.npy`). On a re-run, finished sources are skipped but their fingerprints are loaded, so cross-source duplicate removal still works.


In [3]:
SOURCES = {                    # processed in this order: earlier sources win when a duplicate appears twice
    "sc":        {"file": "train_supreme_court.json", "min_words_raw": 0},
    "tribunals": {"file": "train_tribunals.json",     "min_words_raw": 0},
    "acts":      {"file": "train_acts.json",          "min_words_raw": 0},
    "hc_long":   {"file": "train_high_courts.json",   "min_words_raw": 640},
}
MIN_WORDS_CLEAN = 50           # drop empty / near-empty documents after cleaning
EVAL_MIN_WORDS = 700           # SC validation docs long enough for a 512 + 128 example (with margin)

seen = set()                   # fingerprints of every document kept so far (all sources)

def fingerprint(text):
    return int.from_bytes(hashlib.md5(text.encode("ascii")).digest()[:8], "little")

def process_source(name, cfg):
    meta_path = OUT_DIR / f"{name}_meta.json"
    hash_path = OUT_DIR / f"{name}_hashes.npy"
    if meta_path.exists():                                   # finished earlier -> skip, but remember its fingerprints
        seen.update(np.load(hash_path).tolist())
        meta = json.load(open(meta_path))
        print(f"[cache] {name}: already done ({meta['train_tokens']/1e6:,.0f}M train tokens) - skipping")
        return meta

    raw_path = hf_hub_download(REPO_ID, f"Phase_1_Pretraining/Train/{cfg['file']}", repo_type="dataset")
    raw_size = os.path.getsize(raw_path)

    tmp_train = open(OUT_DIR / f"{name}_train.bin.tmp", "wb")
    tmp_val   = open(OUT_DIR / f"{name}_val.bin.tmp", "wb")
    stats = dict(docs_read=0, filtered_raw=0, too_short=0, duplicates=0, train_docs=0, val_docs=0,
                 train_tokens=0, val_tokens=0, train_words=0, val_words=0)
    kept_hashes, eval_docs = [], []
    batch_texts, batch_val = [], []

    def flush():
        # tokenize the collected batch and append it to the train/val files
        for enc, text, is_val in zip(tokenizer.encode_batch(batch_texts), batch_texts, batch_val):
            arr = np.array(enc.ids + [EOS_ID], dtype=np.uint16)
            (tmp_val if is_val else tmp_train).write(arr.tobytes())
            side = "val" if is_val else "train"
            stats[f"{side}_tokens"] += len(arr); stats[f"{side}_words"] += len(text.split())
        batch_texts.clear(); batch_val.clear()

    with open(raw_path, "rb") as f, tqdm(total=raw_size, unit="B", unit_scale=True, desc=name) as bar:
        for rec in ijson.items(f, "item"):
            stats["docs_read"] += 1
            if (rec.get("word_count") or 10**9) < cfg["min_words_raw"]:
                stats["filtered_raw"] += 1
            else:
                text = clean_text(rec.get("text") or "")
                n_words = len(text.split())
                if n_words < MIN_WORDS_CLEAN:
                    stats["too_short"] += 1
                else:
                    h = fingerprint(text)
                    if h in seen:
                        stats["duplicates"] += 1
                    else:
                        seen.add(h); kept_hashes.append(h)
                        is_val = (h % 1000) < VAL_PER_MILLE
                        stats["val_docs" if is_val else "train_docs"] += 1
                        if is_val and name == "sc" and n_words >= EVAL_MIN_WORDS:
                            eval_docs.append(text)
                        batch_texts.append(text); batch_val.append(is_val)
                        if len(batch_texts) >= BATCH_DOCS:
                            flush()
            if stats["docs_read"] % 500 == 0:
                bar.update(f.tell() - bar.n)                 # bytes read so far
                bar.set_postfix(train_tok=f"{stats['train_tokens']/1e6:.0f}M")
        flush()
        bar.update(raw_size - bar.n)

    tmp_train.close(); tmp_val.close()
    os.replace(OUT_DIR / f"{name}_train.bin.tmp", OUT_DIR / f"{name}_train.bin")
    os.replace(OUT_DIR / f"{name}_val.bin.tmp",   OUT_DIR / f"{name}_val.bin")
    np.save(hash_path, np.array(kept_hashes, dtype=np.uint64))
    if name == "sc":
        with open(EVAL_DIR / "sc_val_docs.jsonl", "w", encoding="ascii") as ef:
            for t in eval_docs:
                ef.write(json.dumps({"text": t}) + "\n")
    stats["tokens_per_word"] = (stats["train_tokens"] + stats["val_tokens"]) / max(stats["train_words"] + stats["val_words"], 1)
    json.dump(stats, open(meta_path, "w"), indent=2)         # written LAST = "this source is complete"
    return stats

## 3. Run it for all four sources

Rough timing for the first run: downloads ~8 min in total (the HF progress bars), then parsing + cleaning + tokenizing.
SC and Acts are quick; Tribunals and High Courts take longest. Expect **roughly 30–60 min** overall. Re-runs skip finished sources.

The laptop GPU is not used here (this is CPU work), so you can keep working in another notebook.


In [4]:
t0 = time.time()
all_stats = {}
for name, cfg in SOURCES.items():
    all_stats[name] = process_source(name, cfg)
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

sc: 100%|██████████| 1.32G/1.32G [04:09<00:00, 5.29MB/s, train_tok=251M]
c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Atharva\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Atharva\Desktop\Assignments\LL


Done in 46.4 min


## 4. Summary table (for the report)


In [5]:
print(f"{'source':10s} {'read':>9s} {'filtered':>9s} {'<50 w':>7s} {'dups':>6s} {'train docs':>11s} {'val docs':>9s} "
      f"{'train tok':>10s} {'val tok':>8s} {'tok/word':>9s}")
tot_train = tot_val = 0
for name, s in all_stats.items():
    print(f"{name:10s} {s['docs_read']:9,d} {s['filtered_raw']:9,d} {s['too_short']:7,d} {s['duplicates']:6,d} "
          f"{s['train_docs']:11,d} {s['val_docs']:9,d} {s['train_tokens']/1e6:9.0f}M {s['val_tokens']/1e6:7.1f}M "
          f"{s['tokens_per_word']:9.3f}")
    tot_train += s["train_tokens"]; tot_val += s["val_tokens"]
print(f"\nTOTAL: {tot_train/1e9:.2f}B train tokens, {tot_val/1e6:.0f}M validation tokens")
json.dump(all_stats, open(OUT_DIR / "summary.json", "w"), indent=2)

source          read  filtered   <50 w   dups  train docs  val docs  train tok  val tok  tok/word
sc            55,549         0     108    157      54,730       554       252M     2.8M     1.217
tribunals    869,132         0   3,251  4,796     852,573     8,512      1404M    13.8M     1.484
acts          18,826         0       9     91      18,533       193       149M     1.6M     1.261
hc_long    2,536,738 2,045,265       0 76,801     410,546     4,126       931M     9.5M     1.330

TOTAL: 2.74B train tokens, 28M validation tokens


## 5. Sanity check: read the token file back

`np.memmap` opens a `.bin` file as an array **without loading it into RAM** — it reads only the parts you touch.
This is also how the architecture notebooks will read their training data.
We check: size matches the summary, the number of EOS tokens equals the number of documents, and the first tokens decode to readable, cleaned text.


In [6]:
sc_train = np.memmap(OUT_DIR / "sc_train.bin", dtype=np.uint16, mode="r")
print(f"sc_train.bin: {len(sc_train):,} tokens (summary says {all_stats['sc']['train_tokens']:,})")
print(f"EOS tokens  : {int((sc_train == EOS_ID).sum()):,} (train docs: {all_stats['sc']['train_docs']:,})")
print("\nFirst 120 tokens decoded:\n", tokenizer.decode(sc_train[:120].tolist()))

sc_train.bin: 252,108,629 tokens (summary says 252,108,629)
EOS tokens  : 54,730 (train docs: 54,730)

First 120 tokens decoded:
 PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs. RESPONDENT: THE STATE OF MYSORE DATE OF JUDGMENT: 18/12/1961 ACT: Temple-Large number of pilgrims visiting and worshiping-If private or public-Madras Temple Entry Authorisation Act, 1947(Mad. V. of 1947), s. 2(1). HEADNOTE: The shrine in suit which was originally founded by Shri Madhavacharya, a Hindu saint was managed for a long time in rotation by the heads of eight


## 6. New local evaluation set (from cleaned SC validation documents)

Why a new set: the old one came from the _baseline's_ split. Some of those documents may now be in our **training** data, which would make local scores look too good.
The new set comes from our hash-based validation split, so none of its documents are ever trained on, and it is cleaned exactly like the test prompts.

Same recipe as before: random start (seeded), 512 words prompt, next 128 words truth. Formats match `test.json` and `evaluate_phase1.py`.
Skipped if the files already exist.


In [7]:
PROMPTS_PATH = EVAL_DIR / "local_eval_prompts.json"
TRUTH_PATH   = EVAL_DIR / "local_eval_truth.json"
N_EVAL = 200

if PROMPTS_PATH.exists() and TRUTH_PATH.exists():
    print("[cache] local evaluation set exists - skipping")
else:
    docs = [json.loads(line)["text"] for line in open(EVAL_DIR / "sc_val_docs.jsonl", encoding="ascii")]
    rng = random.Random(SEED)
    rng.shuffle(docs)
    prompts, truth = {}, {}
    for i, text in enumerate(docs[:N_EVAL]):
        words = text.split()
        start = rng.randint(0, len(words) - 640)
        prompts[str(i)] = {"text": " ".join(words[start:start + 512])}
        truth[str(i)]   = {"text": " ".join(words[start + 512:start + 640])}
    json.dump(prompts, open(PROMPTS_PATH, "w"), indent=1)
    json.dump(truth,   open(TRUTH_PATH, "w"), indent=1)
    print(f"Saved {len(prompts)} pairs (from {len(docs)} long SC validation docs)")
p = json.load(open(PROMPTS_PATH)); t = json.load(open(TRUTH_PATH))
print("Example prompt end:", "..." + p["0"]["text"][-150:])
print("Example truth     :", t["0"]["text"][:150], "...")

Saved 200 pairs (from 471 long SC validation docs)
Example prompt end: ...of Section 3(15) of the Estate Land Act? The said section reads as under: "3(15). "Ryot" means a person who holds for the purpose of agriculture ryoti
Example truth     : land in an estate on condition of paying to the landholder the rent which is legally due upon it." Section 3 (16) defines ryot land means cultivable l ...


## Viva check

1. Why is a hash-based train/val split more robust than "5% of documents in file order"?
2. Why do we remove duplicates _before_ splitting?
3. Why do we filter High Court documents by `word_count` _before_ cleaning them?
4. What does `np.memmap` do, and why does it matter for a multi-GB token file on a 16 GB laptop?
5. Why did we build a new local evaluation set instead of reusing the baseline's?
6. Why must every source use the same tokenizer and the same cleaning?
